# E-Commerce RAG: GPU Embedder
Bu notebook, 5.000 ürün ve yorumlarının `BAAI/bge-m3` modeli ile saniyeler içinde vektörize edilmesi için hazırlanmıştır.

**Kullanım:**
1. Menüden **Runtime > Change runtime type** seçin ve **T4 GPU**'yu işaretleyin.
2. Sol taraftaki klasör simgesine tıklayıp, bilgisayarınızdaki `sample_products.jsonl` ve `sample_reviews.jsonl` dosyalarını yükleyin.
3. Tüm hücreleri sırayla çalıştırın (Runtime > Run all).
4. İşlem bitince soldaki menüden `embedded_chunks.jsonl` dosyasını bilgisayarınıza indirin.

In [ ]:
!pip install sentence-transformers


In [ ]:
import json
import os
import time
from collections import defaultdict
import torch
from sentence_transformers import SentenceTransformer

PRODUCTS_FILE = 'sample_products.jsonl'
REVIEWS_FILE = 'sample_reviews.jsonl'
OUTPUT_FILE = 'embedded_chunks.jsonl'

MAX_REVIEWS_PER_PRODUCT = 5
MIN_REVIEW_LENGTH = 50


In [ ]:
# --- CHUNKING ALGORİTMALARI ---
def make_metadata_chunk(product: dict) -> str:
    title = product.get('title', 'Unknown Product')
    category = ' > '.join(product.get('categories', [])) or product.get('category', 'Unknown')
    price = product.get('price')
    price_str = f'${price:.2f}' if price else 'Price not listed'
    avg_rating = product.get('average_rating', 0.0)
    details = product.get('details', {})
    brand = details.get('Brand') or details.get('brand') or 'Unknown brand'
    
    raw = (f'Product: {title}\nCategory: {category}\nBrand: {brand}\n'
           f'Price: {price_str}\nRating: {avg_rating}/5')
    return f'[{title} — metadata]: {raw}'

def recursive_split(text: str, max_chars: int = 500, overlap: int = 50) -> list:
    separators = ['\n\n', '\n', '. ', ', ', ' ', '']
    def _split(text: str, seps: list) -> list:
        if not seps:
            return [text[i:i + max_chars] for i in range(0, len(text), max_chars - overlap)]
        sep = seps[0]
        parts = text.split(sep) if sep else list(text)
        chunks, current = [], ''
        for part in parts:
            candidate = current + (sep if current else '') + part
            if len(candidate) <= max_chars:
                current = candidate
            else:
                if current:
                    chunks.append(current.strip())
                if len(part) > max_chars:
                    chunks.extend(_split(part, seps[1:]))
                    current = ''
                else:
                    current = part
        if current.strip():
            chunks.append(current.strip())
        return chunks
    
    raw_chunks = _split(text.strip(), separators)
    overlapped = []
    for i, chunk in enumerate(raw_chunks):
        if i > 0 and overlap > 0:
            chunk = raw_chunks[i - 1][-overlap:] + ' ' + chunk
        overlapped.append(chunk.strip())
    return [c for c in overlapped if len(c) > 20]

def make_description_chunks(product: dict) -> list:
    title = product.get('title', 'Unknown Product')
    features = product.get('features', []) or []
    description = product.get('description', []) or []
    parts = [str(f) for f in features if str(f).strip()] + [str(d) for d in description if str(d).strip()]
    full_text = '\n'.join(parts).strip()
    if len(full_text) < 30: return []
    raw_chunks = recursive_split(full_text)
    return [f'[{title} — description]: {chunk}' for chunk in raw_chunks]

def make_review_chunk(product_title: str, review_title: str, review_text: str) -> str:
    header = f'Review title: {review_title}' if review_title else ''
    body = review_text.strip()
    raw = f'{header}\n{body}'.strip() if header else body
    return f'[{product_title} — review]: {raw}'


In [ ]:
# --- VERİLERİ OKUMA VE CHUNK OLUŞTURMA ---
if not os.path.exists(PRODUCTS_FILE) or not os.path.exists(REVIEWS_FILE):
    raise FileNotFoundError('Lütfen sol taraftaki menüden sample_products.jsonl ve sample_reviews.jsonl dosyalarını yükleyin!')

print('1. Ürünler yükleniyor...')
raw_products = []
with open(PRODUCTS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): raw_products.append(json.loads(line))

products_by_asin = {p['parent_asin']: p for p in raw_products if p.get('parent_asin')}
target_asins = set(products_by_asin.keys())

chunk_records = []
for raw in raw_products:
    asin = raw.get('parent_asin')
    if not asin: continue
    
    chunk_records.append({'sku': asin, 'type': 'metadata', 'index': 0, 'content': make_metadata_chunk(raw)})
    
    for idx, content in enumerate(make_description_chunks(raw)):
        chunk_records.append({'sku': asin, 'type': 'description', 'index': idx, 'content': content})

print('2. Yorumlar yükleniyor (Erken durdurma devrede)...')
reviews_by_asin = defaultdict(list)
full_asins = set()
BUFFER = MAX_REVIEWS_PER_PRODUCT * 3

with open(REVIEWS_FILE, 'r', encoding='utf-8') as f:
    for lines_read, line in enumerate(f, 1):
        if lines_read % 1_000_000 == 0:
            if len(full_asins) >= len(target_asins):
                break
        if 'parent_asin' not in line: continue
        try:
            review = json.loads(line)
        except: continue
        
        asin = review.get('parent_asin')
        if not asin or asin not in target_asins or asin in full_asins: continue
        if len((review.get('text') or '').strip()) < MIN_REVIEW_LENGTH: continue
            
        reviews_by_asin[asin].append(review)
        if len(reviews_by_asin[asin]) >= BUFFER:
            full_asins.add(asin)

for asin, reviews in reviews_by_asin.items():
    reviews.sort(key=lambda r: r.get('helpful_vote', 0), reverse=True)
    product_title = (products_by_asin[asin].get('title') or 'Unknown')[:200]
    for idx, review in enumerate(reviews[:MAX_REVIEWS_PER_PRODUCT]):
        content = make_review_chunk(product_title, review.get('title', ''), review.get('text', ''))
        chunk_records.append({'sku': asin, 'type': 'review', 'index': idx, 'content': content})

print(f'Toplam oluşturulan chunk sayısı: {len(chunk_records)}')


In [ ]:
# --- GPU İLE EMBEDDING ---
print('3. BAAI/bge-m3 Modeli yükleniyor...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Kullanılan cihaz: {device.upper()}')

model = SentenceTransformer('BAAI/bge-m3', device=device)

print('4. Embedding işlemi başlıyor...')
t0 = time.time()

texts = [r['content'] for r in chunk_records]
# GPU üzerinde devasa batch'lerle işleyebiliriz
embeddings = model.encode(texts, batch_size=256, show_progress_bar=True, convert_to_numpy=True)

print(f'\nSüre: {time.time() - t0:.1f} saniye')

print('5. Sonuçlar dosyaya kaydediliyor...')
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for record, emb in zip(chunk_records, embeddings):
        record['embedding'] = emb.tolist()
        f.write(json.dumps(record) + '\n')

print(f'\n✅ İŞLEM TAMAMLANDI! Sol menüden {OUTPUT_FILE} dosyasını indirebilirsiniz.')
